In [29]:
from sklearn.metrics import f1_score, precision_recall_fscore_support, roc_curve, auc
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
from scipy.stats import norm
import lightgbm as lgb
import xgboost as xgb
import pandas as pd
import numpy as np
from tabpfn import TabPFNClassifier

In [30]:
df = pd.read_parquet('/home/izadoraganem/Mineracao/df_total.parquet')

In [ ]:
print(df[df['obito'] == 1])

      idade  sexo  scv___1  scv___2  scv___3  scv___4  scv___5  scv___6  \
4      77.0   1.0        1        0        0        0        0        0   
8      66.0   2.0        1        0        0        0        0        0   
11     82.0   2.0        1        1        1        1        0        0   
52     67.0   1.0        1        0        0        0        0        0   
56     75.0   2.0        1        0        0        0        0        0   
...     ...   ...      ...      ...      ...      ...      ...      ...   
6229   61.0   2.0        1        0        0        0        0        0   
6230   68.0   2.0        1        0        0        0        0        0   
6231   84.0   1.0        1        0        0        0        0        0   
6232   68.0   1.0        1        0        0        0        0        0   
6234   70.0   2.0        1        0        0        0        0        0   

      sr___1  sr___2  ...  vacina  dimero_adm_final_relacao  Hometown_GDP  \
4          0       1  

In [31]:
def IC_95(medidas):
    media = np.mean(medidas)
    erro_padrao = np.std(medidas, ddof=1) / np.sqrt(len(medidas))
    intervalo = norm.ppf(0.975) * erro_padrao  # z-score 95%
    return f"{media:.4f}({intervalo:.4f})"

def IC_95_percentage(medidas):
    media = np.mean(medidas)*100
    erro_padrao = np.std(medidas, ddof=1) / np.sqrt(len(medidas))*100
    intervalo = norm.ppf(0.975) * erro_padrao
    return f"{media:.1f}({intervalo:.1f})"

def evaluateModel(y_true_folds, y_pred_folds, classes, modelo_nome):
    macro_f1s, micro_f1s, precisao_folds, recall_folds, f1_folds = [], [], [], [], []
    
    for y_true, y_pred in zip(y_true_folds, y_pred_folds):
        macro_f1s.append(f1_score(y_true, y_pred, average="macro"))
        micro_f1s.append(f1_score(y_true, y_pred, average="micro"))
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=None, labels=classes, zero_division=0
        )
        precisao_folds.append(prec)
        recall_folds.append(rec)
        f1_folds.append(f1)

    precisao_folds = np.array(precisao_folds)
    recall_folds = np.array(recall_folds)
    f1_folds = np.array(f1_folds)

    resultados = {
        "modelo": modelo_nome,
        "macro_f1": IC_95_percentage(macro_f1s),
        "micro_f1": IC_95_percentage(micro_f1s)
    }

    for i, c in enumerate(classes):
        resultados[f"precisao_{c}"] = IC_95_percentage(precisao_folds[:, i])
        resultados[f"recall_{c}"] = IC_95_percentage(recall_folds[:, i])
        resultados[f"f1_{c}"] = IC_95_percentage(f1_folds[:, i])
    
    return resultados

def trainModels(df, target_col, classificador):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    classes = np.unique(y)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    y_true_folds, y_pred_folds = [], []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        if classificador == "LightGBM":
            modelo = lgb.LGBMClassifier(random_state=7, verbose=-1)
        elif classificador == "XGBoost":
            modelo = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=7)
        elif classificador == "TabPFN":
            modelo = TabPFNClassifier(device='cuda', random_state=7)
        else:
            raise ValueError(f"Classificador '{classificador}' não reconhecido.")

        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

        y_true_folds.append(y_test)
        y_pred_folds.append(y_pred)

    resultados = evaluateModel(y_true_folds, y_pred_folds, classes, classificador)
    return resultados



In [32]:
FEATURE_NAME_MAP = {
    "idade": "age",
    "sexo": "sex",

    "scv___1": "cardiovascular_disease_1",
    "scv___2": "cardiovascular_disease_2",
    "scv___3": "cardiovascular_disease_3",
    "scv___4": "cardiovascular_disease_4",
    "scv___5": "cardiovascular_disease_5",
    "scv___6": "cardiovascular_disease_6",

    "sr___1": "systemic_response_1",
    "sr___2": "systemic_response_2",
    "sr___3": "systemic_response_3",

    "dm___1": "diabetes_mellitus_1",
    "dm___2": "diabetes_mellitus_2",

    "comorb_outras___1": "other_comorbidities_1",
    "comorb_outras___3": "other_comorbidities_3",
    "comorb_outras___4": "other_comorbidities_4",
    "comorb_outras___5": "other_comorbidities_5",
    "comorb_outras___6": "other_comorbidities_6",
    "comorb_outras___8": "other_comorbidities_8",
    "comorb_outras___11": "other_comorbidities_11",
    "comorb_outras_dialise": "dialysis_dependent_comorbidity",

    "med_domiciliar___3": "home_medication_3",
    "med_domiciliar___6": "home_medication_6",
    "med_domiciliar___7": "home_medication_7",
    "med_domiciliar___13": "home_medication_13",

    "hv___1": "clinical_history_1",
    "hv___2": "clinical_history_2",
    "hv___3": "clinical_history_3",
    "hv___4": "clinical_history_4",

    "glasgow_menor15_adm_final": "glasgow_coma_scale_less_than_15",

    "fc_adm_final": "heart_rate",
    "fr_adm_final": "respiratory_rate",
    "temp_adm_final": "body_temperature",
    "sat_fio2": "oxygen_saturation_fio2_ratio",
    "vm_adm_final": "mechanical_ventilation",

    "hb_adm_final": "hemoglobin",
    "leucocitos_adm_final": "leukocytes",
    "neutrofilos_adm_final": "neutrophils",
    "linfocitos_adm_final": "lymphocytes",
    "plaquetas_adm_final": "platelets",
    "creatinina_adm_final": "creatinine",
    "lactato_adm_final_padrao": "lactate",
    "pcr_adm_final": "c_reactive_protein",
    "sodio_adm_final": "sodium",
    "ast_alt": "ast_alt_ratio",
    "ureia_adm_final": "urea",
    "ph_adm_final": "blood_ph",
    "pco2_adm_final": "pco2",
    "bicarbonato_adm_final": "bicarbonate",
    "po2_fio2_adm_final": "pao2_fio2_ratio",
    "dimero_adm_final_relacao": "d_dimer_ratio",

    "pasamina90": "systolic_bp_below_90",
    "padamina60": "diastolic_bp_below_60",

    "vacina": "vaccination_status",

    "Hometown_GDP": "hometown_gdp",
    "Hometown_DHI": "hometown_human_development_index",
    "Hospital_GDP": "hospital_region_gdp",
    "Hospital_DHI": "hospital_region_human_development_index",
    "Academic_status": "hospital_academic_status",
    "Accreditation": "hospital_accreditation",
    "Source_of_income": "source_of_income",

    "obito": "mortality",
}

In [ ]:
classificadores = ["LightGBM", "XGBoost", "TabPFN"]

resultados_gerais = []

for metodo in classificadores:
    resultados = trainModels(df, target_col='obito', classificador=metodo)
    resultados_gerais.append(resultados)

df_resultados = pd.DataFrame(resultados_gerais)
df_resultados.to_csv('resultados_modelos.csv', index=False)

print(df_resultados)

Treinando e avaliando: LightGBM...
Treinando e avaliando: XGBoost...


/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [22:49:27] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [22:49:27] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [22:49:27] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [22:49:27] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain,

Treinando e avaliando: TabPFN...

Resultados finais:
     modelo   macro_f1   micro_f1 precisao_0.0 recall_0.0     f1_0.0  \
0  LightGBM  89.4(1.0)  94.5(0.5)    95.4(0.4)  98.1(0.3)  96.8(0.3)   
1   XGBoost  90.2(1.0)  94.8(0.6)    95.9(0.3)  98.0(0.5)  96.9(0.3)   
2    TabPFN  98.6(0.4)  99.2(0.2)    99.5(0.2)  99.6(0.3)  99.5(0.1)   

  precisao_1.0 recall_1.0     f1_1.0  
0    89.1(1.9)  76.2(2.4)  82.1(1.8)  
1    88.7(2.6)  78.7(1.8)  83.4(1.8)  
2    98.0(1.4)  97.3(0.8)  97.6(0.6)  


In [ ]:
def analisar_divergencias_shap(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    casos_divergentes = []
    total_samples = 0
    total_disagreements = 0
    tab_correct_in_disagreement = 0
    xgb_correct_in_disagreement = 0

    for fold_i, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        total_samples += len(X_test)

        model_xgb = xgb.XGBClassifier(
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=7
        )
        model_xgb.fit(X_train, y_train)
        pred_xgb = model_xgb.predict(X_test)

        model_tab = TabPFNClassifier(
            device="cuda",
            ignore_pretraining_limits=True,
            random_state=7
        )
        model_tab.fit(X_train, y_train)
        pred_tab = model_tab.predict(X_test)

        disagreement_mask = pred_tab != pred_xgb

        total_disagreements += np.sum(disagreement_mask)

    
        tab_correct_mask = (pred_tab == y_test)
        xgb_correct_mask = (pred_xgb == y_test)

        tab_correct_in_disagreement += np.sum(disagreement_mask & tab_correct_mask)
        xgb_correct_in_disagreement += np.sum(disagreement_mask & xgb_correct_mask)

  
        indices_divergentes = X_test.index[disagreement_mask].tolist()

        for idx in indices_divergentes:
            casos_divergentes.append({
                "fold": fold_i,
                "original_index": idx,
                "true_label": y_test.loc[idx],
                "xgb_pred": pred_xgb[np.where(X_test.index == idx)[0][0]],
                "tab_pred": pred_tab[np.where(X_test.index == idx)[0][0]],
                "row_data": X_test.loc[[idx]],
                "model_tab_instance": model_tab,
                "background_data": X_train.sample(50, random_state=42),
            })

    if total_disagreements > 0:
        disagreement_rate = total_disagreements / total_samples
        tab_accuracy_disagreement = tab_correct_in_disagreement / total_disagreements
        xgb_accuracy_disagreement = xgb_correct_in_disagreement / total_disagreements
    else:
        disagreement_rate = 0
        tab_accuracy_disagreement = 0
        xgb_accuracy_disagreement = 0

    print("\n===== RESULTADOS GLOBAIS =====")
    print(f"Total de amostras: {total_samples}")
    print(f"Total de divergências: {total_disagreements}")
    print(f"TabPFN correto nas divergências: {tab_correct_in_disagreement}")
    print(f"XGBoost correto nas divergências: {xgb_correct_in_disagreement}")


    print("\n----- Percentuais -----")
    print(f"Disagreement rate total (x%): {disagreement_rate*100:.2f}%")
    print(f"Disagreement rate obito: {total_disagreements/1035*100:.2f}%")
    print(f"TabPFN accuracy em divergências (y%): {tab_accuracy_disagreement*100:.2f}%")
    print(f"XGBoost accuracy em divergências (z%): {xgb_accuracy_disagreement*100:.2f}%")

    return {
        "casos_divergentes": casos_divergentes,
        "metrics": {
            "total_samples": total_samples,
            "total_disagreements": total_disagreements,
            "tab_correct_in_disagreement": tab_correct_in_disagreement,
            "xgb_correct_in_disagreement": xgb_correct_in_disagreement,
            "disagreement_rate": disagreement_rate,
            "tab_accuracy_disagreement": tab_accuracy_disagreement,
            "xgb_accuracy_disagreement": xgb_accuracy_disagreement,

        }
    }

In [49]:
resultado = analisar_divergencias_shap(df, target_col="obito")

casos = resultado["casos_divergentes"]
metricas = resultado["metrics"]

print(metricas)

Iniciando análise completa de divergências...


/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [23:06:58] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [23:07:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [23:07:04] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [23:07:07] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain,


===== RESULTADOS GLOBAIS =====
Total de amostras: 6237
Total de divergências: 307
Total de óbitos: 1035
TabPFN correto nas divergências: 291
XGBoost correto nas divergências: 16

----- Percentuais -----
Disagreement rate total (x%): 4.92%
Disagreement rate obito: 29.66%
TabPFN accuracy em divergências (y%): 94.79%
XGBoost accuracy em divergências (z%): 5.21%
{'total_samples': 6237, 'total_disagreements': np.int64(307), 'tab_correct_in_disagreement': np.int64(291), 'xgb_correct_in_disagreement': np.int64(16), 'disagreement_rate': np.float64(0.04922238255571589), 'tab_accuracy_disagreement': np.float64(0.9478827361563518), 'xgb_accuracy_disagreement': np.float64(0.05211726384364821)}


In [ ]:
def gerar_tabela_shap_divergencias(casos_divergentes, feature_names, X_train_ref, y_train_ref):

    grupos = {
        "tab_certo_xgb_errado": [],
        "xgb_certo_tab_errado": []
    }
    for caso in casos_divergentes:
        if caso["tab_pred"] == caso["true_label"] and caso["xgb_pred"] != caso["true_label"]:
            grupos["tab_certo_xgb_errado"].append(caso)

        elif caso["xgb_pred"] == caso["true_label"] and caso["tab_pred"] != caso["true_label"]:
            grupos["xgb_certo_tab_errado"].append(caso)

    resultados = {}
    for nome_grupo, casos in grupos.items():

        shap_tab_list = []
        shap_xgb_list = []

        print(f"\nProcessando grupo: {nome_grupo} | n={len(casos)}")

        for caso in casos:

            x = caso["row_data"].values.astype(float)

            model_xgb = caso["model_xgb_instance"]

            explainer_xgb = shap.TreeExplainer(model_xgb)
            shap_xgb = explainer_xgb.shap_values(x)

            if isinstance(shap_xgb, list):
                shap_xgb = shap_xgb[1]  

            shap_xgb_list.append(shap_xgb[0])

           
            model_tab = caso["model_tab_instance"]

            def f(X):
                return model_tab.predict_proba(X)[:, 1]

            explainer_tab = shapiq.Explainer(
                model=f,
                data=X_train_ref.values,
                labels=y_train_ref.values,
                index="SV",
                max_order=1,
                empty_prediction=np.mean(f(X_train_ref.values))
            )

            explainer_tab.imputer.fit(X_train_ref.values[:100])
            explainer_tab.imputer.precompute()

            explanation = explainer_tab(x)
            shap_tab = explanation.values

            shap_tab_list.append(shap_tab)

       
        shap_tab_mean = np.mean(np.array(shap_tab_list), axis=0)
        shap_xgb_mean = np.mean(np.array(shap_xgb_list), axis=0)

        df_tab = pd.DataFrame({
            "feature": feature_names,
            "mean_shap": shap_tab_mean,
            "abs_shap": np.abs(shap_tab_mean)
        }).sort_values("abs_shap", ascending=False)

        df_xgb = pd.DataFrame({
            "feature": feature_names,
            "mean_shap": shap_xgb_mean,
            "abs_shap": np.abs(shap_xgb_mean)
        }).sort_values("abs_shap", ascending=False)

        resultados[nome_grupo] = {
            "tabpfn": df_tab.head(5),
            "xgboost": df_xgb.head(5)
        }

    return resultados

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from tabpfn import TabPFNClassifier
from sklearn.model_selection import StratifiedKFold
import shap
import shapiq

def analisar_divergencias_shap(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    casos_divergentes = []

    total_samples = 0
    total_disagreements = 0
    tab_correct_in_disagreement = 0
    xgb_correct_in_disagreement = 0


    for fold_i, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        total_samples += len(X_test)

    
        model_xgb = xgb.XGBClassifier(
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=7
        )
        model_xgb.fit(X_train, y_train)
        pred_xgb = model_xgb.predict(X_test)

        model_tab = TabPFNClassifier(
            device="cuda", 
            ignore_pretraining_limits=True,
            random_state=7
        )
        model_tab.fit(X_train, y_train)
        pred_tab = model_tab.predict(X_test)

        disagreement_mask = pred_tab != pred_xgb
        total_disagreements += np.sum(disagreement_mask)

        tab_correct_mask = (pred_tab == y_test)
        xgb_correct_mask = (pred_xgb == y_test)

        tab_correct_in_disagreement += np.sum(disagreement_mask & tab_correct_mask)
        xgb_correct_in_disagreement += np.sum(disagreement_mask & xgb_correct_mask)

        indices_divergentes = X_test.index[disagreement_mask].tolist()
        
        test_idx_list = X_test.index.tolist()

        for idx in indices_divergentes:
            pos = test_idx_list.index(idx)
            casos_divergentes.append({
                "fold": fold_i,
                "original_index": idx,
                "true_label": y_test.loc[idx],
                "xgb_pred": pred_xgb[pos],
                "tab_pred": pred_tab[pos],
                "row_data": X_test.loc[[idx]],
                "model_tab_instance": model_tab,
                "model_xgb_instance": model_xgb, 
                "background_data": X_train.sample(min(50, len(X_train)), random_state=42),
            })

    
    print("\n===== RESULTADOS GLOBAIS =====")
    print(f"Total de amostras: {total_samples}")
    print(f"Total de divergências: {total_disagreements}")
    print(f"TabPFN correto nas divergências: {tab_correct_in_disagreement}")
    print(f"XGBoost correto nas divergências: {xgb_correct_in_disagreement}")

    return {
        "casos_divergentes": casos_divergentes,
        "metrics": {
            "total_samples": total_samples,
            "total_disagreements": total_disagreements,
            "tab_accuracy_disagreement": tab_correct_in_disagreement / total_disagreements if total_disagreements > 0 else 0,
            "xgb_accuracy_disagreement": xgb_correct_in_disagreement / total_disagreements if total_disagreements > 0 else 0,
        }
    }



resultado = analisar_divergencias_shap(df, target_col="obito")
casos = resultado["casos_divergentes"]



Iniciando análise de divergências (5-Fold CV)...


/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [23:45:07] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [23:45:09] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [23:45:12] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [23:45:15] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [23:45:18] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain,


===== RESULTADOS GLOBAIS =====
Total de amostras: 6237
Total de divergências: 307
TabPFN correto nas divergências: 291
XGBoost correto nas divergências: 16


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from tabpfn import TabPFNClassifier
from sklearn.model_selection import StratifiedKFold
import shap
from shapiq.explainer import TabularExplainer

def analisar_perfil_erros_especifico(casos_divergentes, X_train_ref):
    
    feature_names = X_train_ref.columns.tolist()
    
    erros_xgb = [c for c in casos_divergentes if c["xgb_pred"] != c["true_label"]]
    erros_tab = [c for c in casos_divergentes if c["tab_pred"] != c["true_label"]]
    
    X_background = X_train_ref.sample(n=min(30, len(X_train_ref)), random_state=42).values

    print(f"Calculando SHAP para {len(erros_xgb)} erros do XGBoost...")
    shap_xgb_list = []
    
    for caso in erros_xgb:
        m_xgb = caso["model_xgb_instance"]
        x_input = caso["row_data"].values.astype(float)
        
        explainer_xgb = shap.TreeExplainer(m_xgb)
        s_xgb = explainer_xgb.shap_values(x_input)
        
        if isinstance(s_xgb, list): s_xgb = s_xgb[1]
        shap_xgb_list.append(s_xgb.flatten())
    
    df_perfil_xgb = pd.DataFrame({
        "feature": feature_names,
        "mean_impact": np.mean(shap_xgb_list, axis=0),
        "abs_impact": np.abs(np.mean(shap_xgb_list, axis=0))
    }).sort_values("abs_impact", ascending=False)

    print(f"Calculando SHAPIQ para {len(erros_tab)} erros do TabPFN...")
    shap_tab_list = []
    
    for caso in erros_tab:
        m_tab = caso["model_tab_instance"]
        x_input = caso["row_data"].values.astype(float)
        
        def model_predict(data):
            if data.ndim == 1: data = data.reshape(1, -1)
            return m_tab.predict_proba(data)[:, 1]

        explainer_tab = TabularExplainer(
            model=model_predict,
            data=X_background,
            index="SV",
            max_order=1
        )
        
        explanation = explainer_tab.explain(x_input.flatten(), budget=2048)
        shap_tab_list.append(explanation.values)

    df_perfil_tab = pd.DataFrame({
        "feature": feature_names,
        "mean_impact": np.mean(shap_tab_list, axis=0),
        "abs_impact": np.abs(np.mean(shap_tab_list, axis=0))
    }).sort_values("abs_impact", ascending=False)

    return df_perfil_xgb, df_perfil_tab

X_ref = df.drop(columns=['obito'])

df_importancia_erro_xgb, df_importancia_erro_tab = analisar_perfil_erros_especifico(
    casos_divergentes=casos, 
    X_train_ref=X_ref
)



print(df_importancia_erro_xgb.head(10))


print(df_importancia_erro_tab.head(10))

Calculando SHAP para 291 erros do XGBoost...
Calculando SHAPIQ para 16 erros do TabPFN...


/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but TabPFNClassifier was fitted with feature names
  warnings.warn(
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/shapiq/imputer/marginal_imputer.py:109: UserWarning: The sample size is larger than the number of data points in the background set. Reducing the sample size to the number of background samples.
  self.init_background(self.data)
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but TabPFNClassifier was fitted with feature names
  warnings.warn(
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but TabPFNClassifier was fitted with feature names
  warnings.warn(
/home/izadoraganem/.pyenv/versi

ValueError: All arrays must be of the same length

In [ ]:
shap_values_xgb = []
shap_values_tab = []

print("Processando XGBoost...")
erros_xgb = [c for c in casos if c["xgb_pred"] != c["true_label"]]
for caso in erros_xgb:
    explainer = shap.TreeExplainer(caso["model_xgb_instance"])
    s = explainer.shap_values(caso["row_data"].values.astype(float))
    if isinstance(s, list): s = s[1]
    shap_values_xgb.append(s.flatten())

print("Processando TabPFN...")
erros_tab = [c for c in casos if c["tab_pred"] != c["true_label"]]
X_background = X_ref.sample(n=30, random_state=42).values
for caso in erros_tab:
    def p(d): return caso["model_tab_instance"].predict_proba(d if d.ndim > 1 else d.reshape(1,-1))[:, 1]
    exp = TabularExplainer(model=p, data=X_background, index="SV", max_order=1)
    explanation = exp.explain(caso["row_data"].values.flatten(), budget=2048)
    shap_values_tab.append(explanation.values)



Processando XGBoost...
Processando TabPFN...


/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but TabPFNClassifier was fitted with feature names
  warnings.warn(
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/shapiq/imputer/marginal_imputer.py:109: UserWarning: The sample size is larger than the number of data points in the background set. Reducing the sample size to the number of background samples.
  self.init_background(self.data)
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but TabPFNClassifier was fitted with feature names
  warnings.warn(
/home/izadoraganem/.pyenv/versions/3.12.7/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but TabPFNClassifier was fitted with feature names
  warnings.warn(
/home/izadoraganem/.pyenv/versi

Cálculos concluídos e salvos em memória!


In [ ]:
def montar_tabelas(lista_xgb, lista_tab, colunas):
    n = len(colunas)
    
    ajustado_xgb = [arr[:n] for arr in lista_xgb]
    ajustado_tab = [arr[:n] for arr in lista_tab]
    
    m_xgb = np.mean(ajustado_xgb, axis=0)
    m_tab = np.mean(ajustado_tab, axis=0)
    
    df_xgb = pd.DataFrame({"feature": colunas, "impact": m_xgb}).sort_values("impact", key=abs, ascending=False)
    df_tab = pd.DataFrame({"feature": colunas, "impact": m_tab}).sort_values("impact", key=abs, ascending=False)
    
    return df_xgb, df_tab

# Uso:
df_final_xgb, df_final_tab = montar_tabelas(shap_values_xgb, shap_values_tab, X_ref.columns.tolist())
display(df_final_xgb)
display(df_final_tab)


,feature,impact
33,sat_fio2,0.631977
39,plaquetas_adm_final,0.523738
38,linfocitos_adm_final,0.103067
44,ast_alt,-0.062472
0,idade,0.057649
...,...,...
19,comorb_outras___11,0.000000
16,comorb_outras___5,0.000000
13,comorb_outras___1,0.000000
18,comorb_outras___8,0.000000


,feature,impact
0,idade,0.166663
37,neutrofilos_adm_final,0.157552
36,leucocitos_adm_final,0.103758
34,vm_adm_final,0.100802
40,creatinina_adm_final,0.079408
...,...,...
28,hv___4,-0.000644
11,dm___1,-0.000609
52,vacina,0.000404
26,hv___2,-0.000330
